# Face Attribute Classifier Training
Training a binary face attribute classifier (Eyeglasses, Hat) using Transfer Learning 
with MobileNetV2 on the CelebA dataset. The output is a TFLite model for on-device 
inference on Android.

> **Note:** Since ran the notebook on Google Colab, the Kaggle API key must be configured manually at runtime.


In [ ]:
import os
import json

kaggle_json = '{"username":"YOUR_USERNAME","key":"YOUR_API_KEY"}'

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

print("Kaggle API key saved")

## 1. Setup
Download the CelebA dataset from Kaggle

In [ ]:
!pip install kaggle -q

!kaggle datasets download -d jessicali9530/celeba-dataset
print("Κατέβηκε!")

## 2. Load Dataset
Extract images and parse attribute annotations. We use only 2 of the 40 CelebA 
attributes: `Eyeglasses` and `Wearing_Hat`.

In [ ]:
import zipfile
import os

print("Unzipping...")
with zipfile.ZipFile("celeba-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("celeba")
print("Done!")

import pandas as pd

df = pd.read_csv("celeba/list_attr_celeba.csv")
df = df.replace(-1, 0)  # Convert CelebA format: -1 → 0 (absent), 1 → 1 (present)

# Extract only the 2 attributes we need
labels = df[['Eyeglasses', 'Wearing_Hat']].values.astype('float32')
image_ids = [f"celeba/img_align_celeba/img_align_celeba/{fname}" for fname in df['image_id']]

print(f"Total images: {len(df)}")
print(f"Eyeglasses distribution: {df['Eyeglasses'].value_counts().to_dict()}")
print(f"Wearing_Hat distribution: {df['Wearing_Hat'].value_counts().to_dict()}")

## 3. Build Model
Transfer learning with MobileNetV2 (ImageNet weights, frozen). We add a small 
classification head with 2 sigmoid outputs.
- Input: 96x96 RGB face crop
- Output: [glasses_prob, hat_prob]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

IMG_SIZE = 96
BATCH_SIZE = 64

X_train, X_val, y_train, y_val = train_test_split(
    image_ids, labels, test_size=0.1, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}") 


train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training is ready to start!")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ]
)
print("Training finished")

## 4. Export to TFLite
Convert the trained Keras model to TFLite format for Android deployment.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('face_attributes.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Model size: {len(tflite_model) / 1024:.1f} KB")
print("Saved as face_attributes.tflite")

# Download
from google.colab import files
files.download('face_attributes.tflite')